In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import numpy as np

def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
import os

image_paths = []
mask_paths = []

# Loop through 'train' and 'val' folders
for split_folder in ['dataset']:
    split_path = os.path.join(path, split_folder)

    # 'images' and 'masks' subfolders exist within each split
    images_folder = os.path.join(split_path, 'images')
    masks_folder = os.path.join(split_path, 'masks')


    for filename in os.listdir(images_folder):
        # Accept common image file extensions for images
        if filename.lower().endswith(('.jpg')):
            image_paths.append(os.path.join(images_folder, filename))

    for filename in os.listdir(masks_folder):
        # Accept common image file extensions for masks
        if filename.lower().endswith(('.png')):
            mask_paths.append(os.path.join(masks_folder, filename))

# Sort paths to ensure images and masks correspond correctly based on filename
image_paths.sort()
mask_paths.sort()

print(f"Total images: {len(image_paths)}")
print(f"Total masks: {len(mask_paths)}")

In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import numpy as np

# Custom Dataset Class
class UnderwaterImagery(Dataset):
  def __init__(self, image_paths, mask_paths, transform=None, target_transform=None):
    self.image_paths = image_paths
    self.mask_paths = mask_paths
    self.transform = transform
    self.target_transform = target_transform

  def __len__(self):
    # Return the number of samples in the dataset
    return len(self.image_paths)

  def __getitem__(self, idx):
    # Load the image and mask at index idx
    image = Image.open(self.image_paths[idx]).convert("RGB")
    mask = Image.open(self.mask_paths[idx]).convert("L")

    # Apply transforms
    if self.transform:
      image = self.transform(image)

    if self.target_transform:
      mask = self.target_transform(mask)
      mask = remap_mask(mask)  # Apply the existing remap_mask for multi-class segmentation

    return image, mask

In [ ]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

# transforms for image and mask
image_transforms = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

mask_transforms = transforms.Compose([
    transforms.Resize((32, 32), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.PILToTensor(),
])

In [ ]:
# Split into train and test sets (80% train, 20% test)
train_images, test_images, train_masks, test_masks = train_test_split(
    image_paths, mask_paths, test_size=0.2, random_state=42
)

# Create Dataset objects
train_dataset = UnderwaterImagery(train_images, train_masks, transform=image_transforms, target_transform=mask_transforms)

test_dataset = UnderwaterImagery(test_images, test_masks, transform=image_transforms, target_transform=mask_transforms)

# Create DataLoaders
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Function to denormalize images
def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])  # ImageNet mean
    std = np.array([0.229, 0.224, 0.225])  # ImageNet std
    img = img.numpy().transpose(1, 2, 0)  # Convert to HWC
    img = img * std + mean  # Reverse normalization
    img = np.clip(img, 0, 1)  # Clip values to [0,1]
    return img

# Display some images with their masks
for i in range(3):
    img, mask = train_dataset[i]
    fig, axes = plt.subplots(1, 2, figsize=(8, 5))
    axes[0].imshow(denormalize(img))
    axes[0].set_title("Image")
    axes[0].axis("off")
    axes[1].imshow(mask.permute(1,2,0), cmap="gray")
    axes[1].set_title("Segmentation Mask")
    axes[1].axis("off")
    plt.show()

In [ ]:
!pip install -q segmentation_models_pytorch

In [ ]:
# TO DO
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define the number of output classes
NUM_CLASSES = 8

model = smp.Unet(
    encoder_name="efficientnet-b1", # choose encoder efficientnet-b1
    encoder_weights="imagenet",     # use `imagenet` pre-trained weights for encoder initialization
    in_channels=3,                  # model input channels
    classes=NUM_CLASSES,            # model output channels
)
model = model.to(device)

In [ ]:
# TO DO
import torch.optim as optim
from tqdm import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device):
  model.train()
  total_loss = 0

  for images, masks in tqdm(dataloader):
    # Move data to device and adjust mask shape/type for CrossEntropyLoss
    images = images.to(device)
    masks = masks.to(device).squeeze(1).long() # Remove channel dimension and convert to Long

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, masks)
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  return total_loss / len(dataloader)

In [ ]:
def validate(model, dataloader, criterion, device):
  model.eval()
  total_loss = 0

  with torch.no_grad():
    for images, masks in dataloader:
            # Move data to device and adjust mask shape/type for CrossEntropyLoss
            images = images.to(device)
            masks = masks.to(device).squeeze(1).long() # Remove channel dimension and convert to Long

            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()

  return total_loss / len(dataloader)

In [ ]:
# TO DO
from torch import nn
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 10 # Train for 10 epochs

In [ ]:
# Run training
train_losses = []
val_losses = []

for epoch in range(num_epochs):
  train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
  val_loss = validate(model, test_loader, criterion, device)

  train_losses.append(train_loss)
  val_losses.append(val_loss)

  print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# TO DO
import random

model.eval()

fig, axes = plt.subplots(4, 3, figsize=(12, 16))

# Get random test samples
indices = random.sample(range(len(test_dataset)), 4)

for i, idx in enumerate(indices):
  image, mask = test_dataset[idx]

  with torch.no_grad():
    image_input = image.unsqueeze(0).to(device)
    prediction_logits = model(image_input)
    # argmax to get the predicted class index for each pixel
    # converts the (1, NUM_CLASSES, H, W) logits to (1, H, W) class indices
    pred = prediction_logits.argmax(dim=1).cpu().squeeze(0) # Squeeze batch dimension

  # Display results
  axes[i, 0].imshow(denormalize(image))
  axes[i, 0].set_title("Underwater Imagery")
  axes[i, 0].axis("off")

  # Ground truth mask should be (H, W) for imshow
  axes[i, 1].imshow(mask.squeeze())
  axes[i, 1].set_title("Ground Truth")
  axes[i, 1].axis("off")

  # Predicted mask be (H, W)
  axes[i, 2].imshow(pred) # Use the same colormap for consistency
  axes[i, 2].set_title("Prediction")
  axes[i, 2].axis("off")

plt.tight_layout()
plt.show()